In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data = pd.read_csv("Sales.csv")

# IDENTIFYING EVERY DATA QUALITY PROBLEM
## PROBLEMS FOUND :
1. Age is in float instead of int
2. Some genders have not been filled
3. Very few users have coupon code
4. Some users dont have a satisfaction score
5. Column names not capitalized
6. Some people's genders were not recorded

In [ ]:
data['gender'].isnull().sum()

In [ ]:
data.info()

# Checking correlation btn nps_score, satisfaction_score and churn

In [ ]:
sns.scatterplot(data=data, x='nps_score', y='satisfaction_score', hue='churn')
plt.plot()

Observation: The above graph shows that low satisfaction score has waaaay more contribution to a customer churning even when their nps score is high

In [ ]:
new_data = data[data['total_spent'] < 2000]
sns.scatterplot(data=new_data, x='total_spent', y='satisfaction_score', hue='churn')
plt.plot()

Observation: Customers with less money spent and low satisfaction score churned very much but for those with high satisfaction score churn iff they have spent little money

In [ ]:
new_data = data[data['total_spent'] < 2000]
sns.scatterplot(data=new_data, x='total_spent', y='nps_score', hue='churn')
plt.plot()

Observation: NPS score seems to be a weak indicator of churn here since even if it varies, churn is still biased towards people who spent less money

# Checking correlation btn gender and churning

In [ ]:
gender_data = data.groupby('gender')['churn'].mean()
sns.barplot(data=data, x='gender', y='churn')
plt.show()

Observation: 'Other' gender have highest churn rates from the above graph

The code below checks number of genders in each gender type

In [ ]:
data['gender'].value_counts()

Filling other genders with unknown 

In [ ]:
data['gender'] = data['gender'].fillna('Unknown')

# Checking for class imbalances

In [ ]:
print("----------- Raw churn data outputs ----------------")
print(data['churn'].value_counts())

print("------------- Percentage breakdown -------------------")
print(data['churn'].value_counts(normalize=True) * 100)

# Dealing with dates

Marekebisho ya tarehe kwenye hizo column za sign_up date and last_purchase_date

In [ ]:
data['signup_date'] = pd.to_datetime(data['signup_date'])
data['last_purchase_date'] = pd.to_datetime(data['last_purchase_date'])

Creating a new feature of days_spent

In [ ]:
data['days_spent'] = (data['last_purchase_date'] - data['signup_date']).dt.days
data.head(20)

The code below kind of shows that the customers with negatve dates and the ones with positive have almost comparable churn rates but the negative days one are a quater of the whole dataset . I choose to discard the days spent column

In [ ]:
neg_days = data[data['days_spent'] <= 0]
pos_days = data[data['days_spent'] > 0]

churn_rate = neg_days['churn'].mean() *100
an_churn_rate = pos_days['churn'].mean() * 100

print(f"Negaive days churn rate is {churn_rate} and Positive days is {an_churn_rate}")

percentage = (len(neg_days) / len(data)) * 100
print(f"Percentage of negative values in your dataset is {percentage} %")


# Dealing with ages and finding correlation btn it and churning

In [ ]:
neg_age = data[data['age'] <= 0].value_counts()
len(neg_age)

Obervation: Some people have negative ages. Removing.........

In [ ]:
data = data[data['age'] > 0]

In [ ]:
age_churn = data.groupby('age')['churn'].mean().reset_index()

sns.scatterplot(data=age_churn, x='age', y='churn')
#sns.lineplot(data=age_churn, x='Age', y='Churn', markers='o', color='red')
plt.xlabel('Age')
plt.ylabel('Churn rate')

plt.show()

Observation: One outlier found. This mzee of 80 years old is skewing my data

# Checking countries and city correlation with churning

In [ ]:
city_churn = data.groupby('city')['churn'].mean().reset_index()
country_churn = data.groupby('country')['churn'].mean().reset_index()

sns.barplot(data=country_churn, x='country', y='churn')
plt.xlabel('Country')
plt.ylabel('Churn rate')

plt.show()

country_churn
# city_churn

Observation: Countries and cities dont seem to contribute to churning in any significant way

The code below checks for the names of columns that we have

In [ ]:
data.columns

# Checking correlation of Device type, Acquisition channel, Subscription type, Is_premium_user, Total_visits, Avg_session_time, Pages_per_session and other categories

Grouping the columns into different categories depending on type of data they have

In [ ]:
items_str = []
items_fig = []
items_none = []

for item in data.columns :
    if isinstance(item, str) :
        items_str.append(item)
    elif isinstance(item, int) or isinstance(item, float):
        items_fig.append(item)
    else :
        items_none.append(item)

In [ ]:
items_fig = list(data.select_dtypes(include=['int64', 'float64']).columns)
items_str = list(data.select_dtypes(include=['object', 'string']).columns)
items_none = [col for col in data.columns if col not in items_fig and col not in items_str]


In [ ]:
len(items_fig) + len(items_str) + len(items_none)

In [ ]:
items_fig

In [ ]:
new_items_str = ['acquisition_channel', 'device_type', 'subscription_type', 'coupon_code', 'payment_method']

new_items_fig = ['total_visits', 'avg_session_time', 'pages_per_session', 'email_open_rate', 
                'email_click_rate', 'avg_order_value', 'discount_used', 'support_tickets', 
                'delivery_delay_days',  'marketing_spend_per_user', 'lifetime_value', 'last_3_month_purchase_freq'
]

Seeing each category relation with churn rate

In [ ]:
for category in new_items_str :
    category_data = data.groupby(category)['churn'].mean()
    display(category_data)

In [ ]:
for category in new_items_fig :
    category_data = data.groupby(category)['churn'].mean()
    display(category_data)

# Checking graphs of other variables

In [ ]:
for category in new_items_fig :
    category_data = data.groupby(category)['churn'].mean().reset_index()

    sns.lineplot(data=category_data, x=category, y='churn', markers='o')
    plt.xlabel(category)
    plt.ylabel("Churn rate")

    plt.show()
    

The following Graphs show denstity estimate plots for churn vs churned values. It helps to see whether churned vs non-churned customers show any statistical variation wrt to each other

In [ ]:
density_est_plot = ['avg_session_time', 'pages_per_session', 'avg_order_value', 'marketing_spend_per_user', 'lifetime_value']
for my_graph in density_est_plot :
    sns.kdeplot(data=data, x=my_graph, hue='churn', common_norm=False)
    plt.show()

Observation: These graphs show that some columns can be dropped since they dont contribute greatly to churn

# Some more graphs and comparisons

From analysis below Refund request can be discarded since churned and unchurned customers have almost equal refund request numbers

In [ ]:
refund_graph = data.groupby('refund_requested')['churn'].mean().reset_index()

display(refund_graph)
sns.lineplot(data=refund_graph, x='refund_requested', y='churn')
plt.xlabel('Refund requested')
plt.ylabel('Churn rate')

plt.show()

In [ ]:
new_data = data[data['age'] > 0]
age_churn = new_data.groupby('age')['churn'].mean().reset_index()

sns.lineplot(data=age_churn, x='age', y='churn', markers='o')

plt.xlabel("Age")
plt.ylabel('Churn rate')

plt.show()

In [ ]:
bins = [0, 18, 25, 35, 45, 55, 65, 100]
labels = ['<18', '18-24', '25-34', '35-44', '45-54', '55-64', '65+']
data['age_group'] = pd.cut(data['age'], bins=bins,labels=labels)

data.groupby('age_group')['churn'].mean()

In [ ]:
print(data['age_group'].value_counts())

Observation: Age seems to be difficult to ignore, it shows some relationship with churning. Hence it will be added to the list predictors, also refund_requested 

# FILTERING DATA

In [ ]:
columns_to_keep = []

columns_to_drop = ['country', 'city','signup_date', 'customer_id', 'last_purchase_date', 
                   'coupon_code', 'delivery_delay_days', 'email_open_rate', 'email_click_rate',
                   'total_visits', 'last_3_month_purchase_freq', 'avg_session_time', 'pages_per_session',
                   'avg_order_value', 'marketing_spend_per_user', 'lifetime_value', 'subscription_type', 
                   'is_premium_user', 'nps_score', 'device_type', 'discount_used',
                   'payment_method', 'acquisition_channel', 'refund_requested', 'days_spent', 'age_group']

In [ ]:
columns_to_keep.extend(col for col in data.columns if col not in columns_to_drop)

In [113]:
columns_to_keep

['gender',
 'age',
 'total_spent',
 'support_tickets',
 'satisfaction_score',
 'churn']

In [ ]:
columns_to_drop

# VERDICT
We have arrived to only 5 key predicators which are 
['gender',
 'age',
 'total_spent',
 'support_tickets',
 'satisfaction_score',
 'churn']

# Saving the new data


In [ ]:
clean_data = data[columns_to_keep]

In [ ]:
clean_data.to_csv('Sales_Clean_Data.csv', index=False)

In [ ]:
clean_data